In [1]:
# %% [markdown]
# # Randomly Initialized Frozen Encoder Evaluation Baseline
# This notebook evaluates a randomly initialized, frozen 1D ResNet encoder backbone 
# coupled with a trained linear head for sleep stage classification on the Sleep-EDF dataset.

# %%
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
import warnings

# Suppress warnings for clean execution logs
warnings.filterwarnings('ignore')

# -------------------------------------------------------------------------
# 1. HARDWARE DEVICE SETUP
# -------------------------------------------------------------------------
# Targeting CUDA device (locked to gpu:1 as per the original setup if available)
DEVICE = torch.device("cuda:1" if torch.cuda.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Evaluating baseline using device: {DEVICE}")

# Import your module-dependent architectures
# Ensure your 'models' folder is in the same directory path as this notebook
from models.models_nc import ResNet1D

# %%
# -------------------------------------------------------------------------
# 2. DOWNSTREAM EVALUATION DATASET
# -------------------------------------------------------------------------
class SleepEDF_Evaluation_Dataset(Dataset):
    def __init__(self, pt_file_path="data/sleep_combined.pt", split="train"):
        """
        Loads the preprocessed Sleep-EDF signals and labels for supervised evaluation.
        """
        print(f"Loading data for evaluation split '{split}'...")
        data_obj = torch.load(pt_file_path, map_location="cpu")
        
        if isinstance(data_obj, dict) and split in data_obj:
            split_data = data_obj[split]
            self.samples = split_data["samples"]
            self.labels = split_data["labels"]
        else:
            raise ValueError(f"Invalid file layout or split '{split}' not found.")
            
        # Ensure samples are FloatTensors
        if not isinstance(self.samples, torch.Tensor):
            self.samples = torch.FloatTensor(self.samples)
        else:
            self.samples = self.samples.float()
            
        # Squeeze down extra channel dimensions
        if self.samples.dim() == 3:
            self.samples = self.samples.squeeze(1) if self.samples.shape[1] == 1 else self.samples.squeeze(2)

        # FIX: ResNet1D internally calls .transpose(-1, -2). 
        # To get [Batch, 1, 3000] inside the model layers, we feed it [Batch, 3000, 1] here.
        self.samples = self.samples.unsqueeze(2) 

        # Convert labels to Long type
        if isinstance(self.labels, torch.Tensor):
            self.labels = self.labels.long()
        else:
            self.labels = torch.tensor(self.labels).long()
        
        print(f"   Loaded {len(self.samples)} windows | Final sample shape: {self.samples.shape}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx], self.labels[idx]

# %%
# -------------------------------------------------------------------------
# 3. CONSTRUCT DATA LOADERS
# -------------------------------------------------------------------------
DATASET_PATH = "/home/gella.saikrishna/code/Learning-with-FrameProjections/data/sleep_combined.pt"

BATCH_SIZE = 128
EPOCHS = 30
LATENT_DIM = 128
NUM_CLASSES = 5  # Sleep-EDF standard classes: W, N1, N2, N3, REM

train_dataset = SleepEDF_Evaluation_Dataset(pt_file_path=DATASET_PATH, split="train")
val_dataset = SleepEDF_Evaluation_Dataset(pt_file_path=DATASET_PATH, split="val")
test_dataset = SleepEDF_Evaluation_Dataset(pt_file_path=DATASET_PATH, split="test")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# %%
# -------------------------------------------------------------------------
# 4. INITIALIZE RANDOM ENCODER AND FREEZE IT
# -------------------------------------------------------------------------
print("\nInitializing fresh, randomly weighted Time Backbone architecture...")
# Instantiating the encoder completely from scratch without loading checkpoints
encoder = ResNet1D(
    in_channels=1, base_filters=32, kernel_size=5, stride=1, groups=1, 
    n_block=3, n_classes=LATENT_DIM, downsample_gap=2, increasefilter_gap=4, \
    use_do=False, backbone=True, output_dim=LATENT_DIM
).to(DEVICE)

print("🔒 Freezing all weights inside the random encoder backbone...")
# Freeze the weights so they act exclusively as static, random feature extractors
for param in encoder.parameters():
    param.requires_grad = False

# Build a clean, trainable Linear Head on top of the frozen outputs
classifier_head = nn.Linear(LATENT_DIM, NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(classifier_head.parameters(), lr=1e-2, weight_decay=1e-4)

# %%
# -------------------------------------------------------------------------
# 5. METRIC TRACKING PIPELINE
# -------------------------------------------------------------------------
def evaluate_model_extended(model_enc, head, data_loader, print_breakdown=False):
    model_enc.eval()
    head.eval()
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for signals, labels in data_loader:
            signals = signals.to(DEVICE)
            
            # Forward pass through the random frozen representations
            _, features = model_enc(signals)
            outputs = head(features)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.numpy())
            
    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average='macro')
    weighted_f1 = f1_score(all_targets, all_preds, average='weighted')
    kappa = cohen_kappa_score(all_targets, all_preds)
    class_f1s = f1_score(all_targets, all_preds, average=None)
    
    if print_breakdown:
        CLASS_NAMES = ["Wake (W)", "N1 Stage", "N2 Stage", "N3 Stage", "REM"]
        print("\n📊 DETAILED PERFORMANCE BREAKDOWN (RANDOM BASELINE):")
        print(f"   ➡️ Test Accuracy:    {acc*100:.2f}%")
        print(f"   ➡️ Cohen's Kappa:     {kappa:.4f}")
        print(f"   ➡️ Macro F1-Score:    {macro_f1:.4f}")
        print(f"   ➡️ Weighted F1-Score: {weighted_f1:.4f}")
        print("   ➡️ Stage-Specific F1-Scores:")
        for name, score in zip(CLASS_NAMES, class_f1s):
            print(f"       • {name.ljust(12)}: {score:.4f}")
            
    return acc, macro_f1

# %%
# -------------------------------------------------------------------------
# 6. EXECUTE SUPERVISED LINEAR HEAD TRAINING
# -------------------------------------------------------------------------
print("\n🏋️ Training Linear Head over Random Features...")
best_val_f1 = 0.0

checkpoint_dir = "checkpoints_baseline"
os.makedirs(checkpoint_dir, exist_ok=True)

for epoch in range(EPOCHS):
    encoder.eval()  # Encoder remains static
    classifier_head.train()
    
    total_loss = 0
    for signals, labels in train_loader:
        signals = signals.to(DEVICE)
        labels = labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        # Pull feature matrices out of the un-updated random network setup
        with torch.no_grad():
            _, features = encoder(signals)
            
        outputs = classifier_head(features)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    val_acc, val_f1 = evaluate_model_extended(encoder, classifier_head, val_loader, print_breakdown=False)
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(classifier_head.state_dict(), os.path.join(checkpoint_dir, "random_baseline_linear_head.pth"))
        
    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc*100:.2f}% | Val Macro-F1: {val_f1:.4f}")

# %%
# -------------------------------------------------------------------------
# 7. FINAL EVALUATION REPORT
# -------------------------------------------------------------------------
print("\n🔒 Final Deployment Testing on Unseen Participants...")
baseline_head_path = os.path.join(checkpoint_dir, "random_baseline_linear_head.pth")
classifier_head.load_state_dict(torch.load(baseline_head_path, map_location=DEVICE))

# Print out comprehensive metric report for the baseline
test_acc, test_f1 = evaluate_model_extended(encoder, classifier_head, test_loader, print_breakdown=True)

Evaluating baseline using device: cuda:1
Loading data for evaluation split 'train'...
   Loaded 25612 windows | Final sample shape: torch.Size([25612, 3000, 1])
Loading data for evaluation split 'val'...
   Loaded 7786 windows | Final sample shape: torch.Size([7786, 3000, 1])
Loading data for evaluation split 'test'...
   Loaded 8910 windows | Final sample shape: torch.Size([8910, 3000, 1])

Initializing fresh, randomly weighted Time Backbone architecture...
🔒 Freezing all weights inside the random encoder backbone...

🏋️ Training Linear Head over Random Features...
Epoch [01/30] | Train Loss: 1.1499 | Val Acc: 62.86% | Val Macro-F1: 0.4514
Epoch [02/30] | Train Loss: 0.9521 | Val Acc: 67.26% | Val Macro-F1: 0.4975
Epoch [03/30] | Train Loss: 0.9165 | Val Acc: 64.22% | Val Macro-F1: 0.4814
Epoch [04/30] | Train Loss: 0.8815 | Val Acc: 66.44% | Val Macro-F1: 0.5023
Epoch [05/30] | Train Loss: 0.8546 | Val Acc: 69.41% | Val Macro-F1: 0.5330
Epoch [06/30] | Train Loss: 0.8465 | Val Acc: 5

In [2]:
# %% [markdown]
# # Randomly Initialized Frozen Encoder Evaluation Baseline (2-Layer Linear Head)
# This notebook evaluates a randomly initialized, frozen 1D ResNet encoder backbone 
# coupled with a 2-layer classification head for sleep stage classification on the Sleep-EDF dataset.

# %%
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
import warnings

# Suppress warnings for clean execution logs
warnings.filterwarnings('ignore')

# -------------------------------------------------------------------------
# 1. HARDWARE DEVICE SETUP
# -------------------------------------------------------------------------
DEVICE = torch.device("cuda:1" if torch.cuda.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Evaluating baseline using device: {DEVICE}")

# Import your module-dependent architectures
from models.models_nc import ResNet1D

# %%
# -------------------------------------------------------------------------
# 2. DOWNSTREAM EVALUATION DATASET
# -------------------------------------------------------------------------
class SleepEDF_Evaluation_Dataset(Dataset):
    def __init__(self, pt_file_path="data/sleep_combined.pt", split="train"):
        """
        Loads the preprocessed Sleep-EDF signals and labels for supervised evaluation.
        """
        print(f"Loading data for evaluation split '{split}'...")
        data_obj = torch.load(pt_file_path, map_location="cpu")
        
        if isinstance(data_obj, dict) and split in data_obj:
            split_data = data_obj[split]
            self.samples = split_data["samples"]
            self.labels = split_data["labels"]
        else:
            raise ValueError(f"Invalid file layout or split '{split}' not found.")
            
        # Ensure samples are FloatTensors
        if not isinstance(self.samples, torch.Tensor):
            self.samples = torch.FloatTensor(self.samples)
        else:
            self.samples = self.samples.float()
            
        # Squeeze down extra channel dimensions
        if self.samples.dim() == 3:
            self.samples = self.samples.squeeze(1) if self.samples.shape[1] == 1 else self.samples.squeeze(2)

        # FIX: ResNet1D internally calls .transpose(-1, -2). 
        # To get [Batch, 1, 3000] inside the model layers, we feed it [Batch, 3000, 1] here.
        self.samples = self.samples.unsqueeze(2) 

        # Convert labels to Long type
        if isinstance(self.labels, torch.Tensor):
            self.labels = self.labels.long()
        else:
            self.labels = torch.tensor(self.labels).long()
        
        print(f"   Loaded {len(self.samples)} windows | Final sample shape: {self.samples.shape}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx], self.labels[idx]

# %%
# -------------------------------------------------------------------------
# 3. CONSTRUCT DATA LOADERS
# -------------------------------------------------------------------------
DATASET_PATH = "/home/gella.saikrishna/code/Learning-with-FrameProjections/data/sleep_combined.pt"

BATCH_SIZE = 128
EPOCHS = 30
LATENT_DIM = 128
HIDDEN_DIM = 64    # Hidden dimension between the 2 linear layers
NUM_CLASSES = 5    # Sleep-EDF standard classes: W, N1, N2, N3, REM

train_dataset = SleepEDF_Evaluation_Dataset(pt_file_path=DATASET_PATH, split="train")
val_dataset = SleepEDF_Evaluation_Dataset(pt_file_path=DATASET_PATH, split="val")
test_dataset = SleepEDF_Evaluation_Dataset(pt_file_path=DATASET_PATH, split="test")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# %%
# -------------------------------------------------------------------------
# 4. INITIALIZE RANDOM ENCODER AND FREEZE IT
# -------------------------------------------------------------------------
print("\nInitializing fresh, randomly weighted Time Backbone architecture...")
encoder = ResNet1D(
    in_channels=1, base_filters=32, kernel_size=5, stride=1, groups=1, 
    n_block=3, n_classes=LATENT_DIM, downsample_gap=2, increasefilter_gap=4, 
    use_do=False, backbone=True, output_dim=LATENT_DIM
).to(DEVICE)

print("🔒 Freezing all weights inside the random encoder backbone...")
for param in encoder.parameters():
    param.requires_grad = False

# -------------------------------------------------------------------------
# 🆕 CHANGED: 2-LAYER CLASSIFICATION HEAD SETUP
# -------------------------------------------------------------------------
classifier_head = nn.Sequential(
    nn.Linear(LATENT_DIM, HIDDEN_DIM),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(HIDDEN_DIM, NUM_CLASSES)
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(classifier_head.parameters(), lr=1e-3, weight_decay=1e-4) # Dropped LR slightly for deep head stability

# %%
# -------------------------------------------------------------------------
# 5. METRIC TRACKING PIPELINE
# -------------------------------------------------------------------------
def evaluate_model_extended(model_enc, head, data_loader, print_breakdown=False):
    model_enc.eval()
    head.eval()
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for signals, labels in data_loader:
            signals = signals.to(DEVICE)
            
            # Forward pass through the random frozen representations
            _, features = model_enc(signals)
            outputs = head(features)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.numpy())
            
    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average='macro')
    weighted_f1 = f1_score(all_targets, all_preds, average='weighted')
    kappa = cohen_kappa_score(all_targets, all_preds)
    class_f1s = f1_score(all_targets, all_preds, average=None)
    
    if print_breakdown:
        CLASS_NAMES = ["Wake (W)", "N1 Stage", "N2 Stage", "N3 Stage", "REM"]
        print("\n📊 DETAILED PERFORMANCE BREAKDOWN (RANDOM BASELINE W/ 2-LAYER HEAD):")
        print(f"   ➡️ Test Accuracy:    {acc*100:.2f}%")
        print(f"   ➡️ Cohen's Kappa:     {kappa:.4f}")
        print(f"   ➡️ Macro F1-Score:    {macro_f1:.4f}")
        print(f"   ➡️ Weighted F1-Score: {weighted_f1:.4f}")
        print("   ➡️ Stage-Specific F1-Scores:")
        for name, score in zip(CLASS_NAMES, class_f1s):
            print(f"       • {name.ljust(12)}: {score:.4f}")
            
    return acc, macro_f1

# %%
# -------------------------------------------------------------------------
# 6. EXECUTE SUPERVISED HEAD TRAINING
# -------------------------------------------------------------------------
print("\n🏋️ Training 2-Layer Linear Head over Random Features...")
best_val_f1 = 0.0

checkpoint_dir = "checkpoints_baseline"
os.makedirs(checkpoint_dir, exist_ok=True)

for epoch in range(EPOCHS):
    encoder.eval()  # Encoder remains static
    classifier_head.train()
    
    total_loss = 0
    for signals, labels in train_loader:
        signals = signals.to(DEVICE)
        labels = labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        with torch.no_grad():
            _, features = encoder(signals)
            
        outputs = classifier_head(features)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    val_acc, val_f1 = evaluate_model_extended(encoder, classifier_head, val_loader, print_breakdown=False)
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(classifier_head.state_dict(), os.path.join(checkpoint_dir, "random_baseline_2layer_head.pth"))
        
    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc*100:.2f}% | Val Macro-F1: {val_f1:.4f}")

# %%
# -------------------------------------------------------------------------
# 7. FINAL EVALUATION REPORT
# -------------------------------------------------------------------------
print("\n🔒 Final Deployment Testing on Unseen Participants...")
baseline_head_path = os.path.join(checkpoint_dir, "random_baseline_2layer_head.pth")
classifier_head.load_state_dict(torch.load(baseline_head_path, map_location=DEVICE))

# Print out comprehensive metric report for the baseline
test_acc, test_f1 = evaluate_model_extended(encoder, classifier_head, test_loader, print_breakdown=True)

Evaluating baseline using device: cuda:1
Loading data for evaluation split 'train'...
   Loaded 25612 windows | Final sample shape: torch.Size([25612, 3000, 1])
Loading data for evaluation split 'val'...
   Loaded 7786 windows | Final sample shape: torch.Size([7786, 3000, 1])
Loading data for evaluation split 'test'...
   Loaded 8910 windows | Final sample shape: torch.Size([8910, 3000, 1])

Initializing fresh, randomly weighted Time Backbone architecture...
🔒 Freezing all weights inside the random encoder backbone...

🏋️ Training 2-Layer Linear Head over Random Features...
Epoch [01/30] | Train Loss: 1.2419 | Val Acc: 57.31% | Val Macro-F1: 0.3372
Epoch [02/30] | Train Loss: 1.0200 | Val Acc: 58.39% | Val Macro-F1: 0.3444
Epoch [03/30] | Train Loss: 0.9241 | Val Acc: 59.72% | Val Macro-F1: 0.3760
Epoch [04/30] | Train Loss: 0.8774 | Val Acc: 67.88% | Val Macro-F1: 0.5038
Epoch [05/30] | Train Loss: 0.8414 | Val Acc: 67.90% | Val Macro-F1: 0.5047
Epoch [06/30] | Train Loss: 0.8269 | Va